# **Tokenize the text**

In [71]:
path = "/content/The_Gift_of_the_Magi.txt"

try:
  with open(path, 'r') as file:
    data = file.read()
  print(data[:50])
except FileNotFoundError:
  print(f"Error: The file '{path}' was not found")

The Gift of the Magi by O. Henry

One dollar and e


In [72]:
print(len(data))

11208


For tokenization the "re" module of python is used

In [73]:
import re

test_tokens_with_spaces = re.split(r'(\s)', data) #capture the whitespaces as tokens
print(f"Tokens with spaces: {test_tokens_with_spaces[:20]}")

test_tokens_without_spaces = re.split(r'\s', data) #whitespaces are used a seperator
print(f"Tokens without spaces: {test_tokens_without_spaces[:20]}")

Tokens with spaces: ['The', ' ', 'Gift', ' ', 'of', ' ', 'the', ' ', 'Magi', ' ', 'by', ' ', 'O.', ' ', 'Henry', '\n', '', '\n', 'One', ' ']
Tokens without spaces: ['The', 'Gift', 'of', 'the', 'Magi', 'by', 'O.', 'Henry', '', 'One', 'dollar', 'and', 'eighty-seven', 'cents.', 'That', 'was', 'all.', 'And', 'sixty', 'cents']


*   Removing whitespaces depends on the task. If the model is sensitive to the exact structure of the text then keeping whitespaces are useful. Eg: Python code
*   For normal tasks removing whitespaces reduces the memory and computation required.





In [74]:
# As the text contains punctuations as well we need to split the tokens taking into account of the punctiations
# tokens = re.split(r'([,.:;?_!"()\'-]|--|\s)', data) #Wrapped in parentheses → capturing group, So the separators are included in the output tokens
tokens = re.split(r'([,.:;?_!"()\'\-–—]|\s)', data)
print(tokens[:30])

['The', ' ', 'Gift', ' ', 'of', ' ', 'the', ' ', 'Magi', ' ', 'by', ' ', 'O', '.', '', ' ', 'Henry', '\n', '', '\n', 'One', ' ', 'dollar', ' ', 'and', ' ', 'eighty', '-', 'seven', ' ']


In [75]:
print(len(tokens))

4811


In [76]:
final_tokens = [t.strip() for t in tokens if t.strip()] #Removes empty tokens along with the leading and trailing spaces in each token
print(final_tokens[:30])

['The', 'Gift', 'of', 'the', 'Magi', 'by', 'O', '.', 'Henry', 'One', 'dollar', 'and', 'eighty', '-', 'seven', 'cents', '.', 'That', 'was', 'all', '.', 'And', 'sixty', 'cents', 'of', 'it', 'was', 'in', 'pennies', '.']


In [77]:
print(f"Total tokens: {len(final_tokens)}")

Total tokens: 2414


# **Converting Tokens to Token IDs**

Each unique token is mapped to an unique integer called Token ID.

We'll use:

*   Set to get unique tokens
*   Dictionay to map tokens to token ids

We'll create a Tokenizer class consisting of two parts encoder and decoder:
* Encoder: Text -> Tokens -> Token Ids
* Decoder: Token Ids -> Tokens -> Text

In [78]:
words = sorted(set(final_tokens))
print(len(words))
print(words)

846
['!', '$1', '$20', '$30', '$8', ',', '-', '.', '87', ':', ';', '?', 'A', 'All', 'Also', 'And', 'As', 'At', 'Babe', 'Be', 'Beautiful', 'Being', 'Broadway', 'But', 'Christmas', 'Combs', 'Coney', 'D', 'Day', 'Dell', 'Della', 'Della’s', 'Dillingham', 'Down', 'Eight', 'Eve', 'Everywhere', 'Expenses', 'For', 'Forget', 'Gift', 'Give', 'God', 'Goods', 'Grand', 'Had', 'Hair', 'He', 'Henry', 'Her', 'His', 'I', 'In', 'Instead', 'Island', 'It', 'It’ll', 'It’s', 'I’m', 'I’ve', 'James', 'Jim', 'Jim’s', 'Kinds', 'King', 'Madame', 'Magi', 'Majesty’s', 'Many', 'Maybe', 'Mr', 'Mrs', 'My', 'Now', 'O', 'Of', 'Oh', 'On', 'Once', 'One', 'Only', 'Out', 'Pennies', 'Perhaps', 'Poor', 'Queen', 'Quietness', 'Rapidly', 'Say', 'Shall', 'She', 'Sheba', 'So', 'Sofronie', 'Solomon', 'Something', 'Suddenly', 'That', 'The', 'Then', 'There', 'They', 'They’re', 'This', 'Three', 'Tomorrow', 'Twenty', 'Watch', 'When', 'Where', 'Which', 'While', 'White', 'With', 'Within', 'You', 'Young', 'Youngs', 'You’ll', 'a', 'able',

In [79]:
vocabulary = {word: id for id, word in enumerate(words)} #A dictionary containing tokens along with their token Ids

for i, word in enumerate(vocabulary.items()):
  print(word)
  if i >= 30:
    break

('!', 0)
('$1', 1)
('$20', 2)
('$30', 3)
('$8', 4)
(',', 5)
('-', 6)
('.', 7)
('87', 8)
(':', 9)
(';', 10)
('?', 11)
('A', 12)
('All', 13)
('Also', 14)
('And', 15)
('As', 16)
('At', 17)
('Babe', 18)
('Be', 19)
('Beautiful', 20)
('Being', 21)
('Broadway', 22)
('But', 23)
('Christmas', 24)
('Combs', 25)
('Coney', 26)
('D', 27)
('Day', 28)
('Dell', 29)
('Della', 30)


This vocabulary contain words from only the given data "The gift of magi". If during encoding we encounter words which are not present in the vocabulary we replace those words with special tokens.

Special Context tokens:
* **<|unk|>** : Used to encode unknown words
* **<|endoftext|>** : Used to seperate different text sources

In [84]:
words.extend(["<|endoftext|>", "<|unk|>"])

vocabulary = {word: id for id, word in enumerate(words)}

for i, item in enumerate(list(vocabulary.items())[-3:]):
  print(item)

('”', 845)
('<|endoftext|>', 854)
('<|unk|>', 855)


In [85]:
class SimpleTokenizerV1:
  def __init__(self, vocabulary):
    self.str_to_int = vocabulary
    self.int_to_str = {id:word for word, id in vocabulary.items()}

  def encode(self, text):
    tokens = re.split(r'([,.:;?_!"()\'\-–—]|\s)', text) # splitting the tokens

    final_tokens = [item.strip() for item in tokens if item.strip()] #Removing spaces
    final_tokens = [item if item in self.str_to_int else "<|unk|>" for item in final_tokens]

    #From the given input text we assign token ids from the vocabulary that we have created
    ids = [self.str_to_int[i] for i in final_tokens]

    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    #Replace spaces before the punctuations in the text
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

Testing the Tokenizer class on random input text

In [87]:
tokenizer = SimpleTokenizerV1(vocabulary)

text = """The magi, as you know, were wise men—wonderfully wise men—who brought
gifts to the Babe in the manger. They invented the art of giving
Christmas presents.
"""
ids1 = tokenizer.encode(text)
print(f'Text 1 encodings: {ids1}')
#Now if a word is not present in the vocabulary it will be replaced with <|unk|> token

text1 = """"Natural Language Processing (NLP) is a fascinating field! It involves breaking down
text into smaller units—tokens—to make it understandable for machines. For instance, isn't Python
a powerful language for this? 'I love coding,' she said, 'especially with AI.'"
"""
ids2 = tokenizer.encode(text1)
print(f"Text 2 encodings: {ids2}")

Text 1 encodings: [98, 467, 5, 157, 811, 433, 5, 775, 791, 477, 814, 797, 791, 477, 814, 785, 187, 340, 730, 710, 18, 409, 710, 471, 7, 101, 417, 710, 156, 523, 343, 24, 577, 7]
Text 2 encodings: [855, 855, 855, 855, 855, 855, 855, 419, 119, 855, 855, 0, 55, 855, 855, 270, 855, 414, 855, 855, 814, 855, 814, 730, 468, 420, 855, 325, 855, 7, 38, 855, 5, 855, 855, 855, 855, 119, 855, 855, 325, 723, 11, 855, 51, 464, 855, 5, 855, 642, 615, 5, 855, 855, 794, 855, 7, 855, 855]


In [88]:
#Testing the decoder function by using the inputs generated by the encoder function
decoded_text = tokenizer.decode(ids1)
decoded_text1 = tokenizer.decode(ids2)
print(f'Decoded text 1: {decoded_text}')
print(f'Decoded text 2: {decoded_text1}')

Decoded text 1: The magi, as you know, were wise men — wonderfully wise men — who brought gifts to the Babe in the manger. They invented the art of giving Christmas presents.
Decoded text 2: <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> is a <|unk|> <|unk|>! It <|unk|> <|unk|> down <|unk|> into <|unk|> <|unk|> — <|unk|> — to make it <|unk|> for <|unk|>. For <|unk|>, <|unk|> <|unk|> <|unk|> <|unk|> a <|unk|> <|unk|> for this? <|unk|> I love <|unk|>, <|unk|> she said, <|unk|> <|unk|> with <|unk|>. <|unk|> <|unk|>


So before passing the input to the encoder we need to add the <|endoftext|> token at the end of every text source.

GPTs use Byte Pair Encoding method to encode tokens and does not use <|unk|> or any other token, only the <|endoftext|> token.